# Runbook de Operacao: operacao-desenvolvimento

Este notebook e obrigatorio para operacoes de desenvolvimento.
Cada celula descreve: o que faz e onde e executada.

Padrao de raiz: `/mnt/repositorio/workdir/repostorios/<repositorio>`.

In [1]:
# O que faz: valida se o repositorio esta no caminho canonico da raiz padrao e mostra o diretorio atual.
# Onde executa: shell local, na raiz do repositorio.
from pathlib import Path
import subprocess

base_root = Path('/mnt/repositorio/workdir/repostorios')
current_root = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip())
repo_name = current_root.name
expected_root = base_root / repo_name

print(f'BASE_ROOT: {base_root}')
print(f'CURRENT_ROOT: {current_root}')
print(f'EXPECTED_ROOT: {expected_root}')

if current_root == expected_root:
    print('[OK] Repo na raiz padrao')
else:
    raise RuntimeError('[ERRO] Repo fora da raiz padrao')

BASE_ROOT: /mnt/repositorio/workdir/repostorios
CURRENT_ROOT: /mnt/repositorio/workdir/repostorios/monorepo-ai-llm
EXPECTED_ROOT: /mnt/repositorio/workdir/repostorios/monorepo-ai-llm
[OK] Repo na raiz padrao


In [2]:
# O que faz: executa checks de organizacao de artefatos do projeto.
# Onde executa: shell local, na raiz do repositorio.
from pathlib import Path
import subprocess

repo_root = Path(subprocess.check_output(['git', 'rev-parse', '--show-toplevel'], text=True).strip())
command = ['node', 'tools/mcp/project-artifacts-organizer.mjs', '--action=check']
print(f'Executando em: {repo_root}')
subprocess.run(command, cwd=repo_root, check=True)

Executando em: /mnt/repositorio/workdir/repostorios/monorepo-ai-llm
[OK] Nenhuma pendencia encontrada (check).


CompletedProcess(args=['node', 'tools/mcp/project-artifacts-organizer.mjs', '--action=check'], returncode=0)

## Autoativacao do .venv no zsh

O que faz: configura o terminal zsh para ativar automaticamente o `.venv` do projeto ao entrar em um diretorio que contenha `.venv/bin/activate`.

Onde executa: shell local do usuario (arquivo `~/.zshrc`).

### Passo a passo

1. Adicionar o bloco abaixo no final do `~/.zshrc`:

```bash
# >>> auto-venv copilot >>>
autoload -U add-zsh-hook

_auto_project_venv() {
  local project_venv="$PWD/.venv"

  if [[ -f "$project_venv/bin/activate" ]]; then
    if [[ "$VIRTUAL_ENV" != "$project_venv" ]]; then
      if [[ -n "$VIRTUAL_ENV" ]]; then
        deactivate 2>/dev/null || true
      fi
      source "$project_venv/bin/activate"
    fi
  else
    if [[ -n "$VIRTUAL_ENV" && "$VIRTUAL_ENV" == */.venv ]]; then
      deactivate 2>/dev/null || true
    fi
  fi
}

add-zsh-hook chpwd _auto_project_venv
_auto_project_venv
# <<< auto-venv copilot <<<
```

2. Validar sintaxe do zsh:

```bash
zsh -n ~/.zshrc
```

3. Recarregar a sessao:

```bash
exec zsh
```

### Validacao funcional

Entrar na raiz do repositorio e confirmar que o `VIRTUAL_ENV` aponta para `<repo>/.venv`:

```bash
cd /mnt/repositorio/workdir/repostorios/monorepo-ai-llm
echo "$VIRTUAL_ENV"
which python
```

Resultado esperado:
- `VIRTUAL_ENV` igual a `/mnt/repositorio/workdir/repostorios/monorepo-ai-llm/.venv`
- `python` resolvendo para `.venv/bin/python`


## MCP para Commit, PR e Sync (via notebook)

O que faz: documenta um fluxo operacional para usar MCP no chat e, quando necessario, comandos locais no notebook para apoiar commit/PR/sync.

Onde executa: chat do agente (MCP) e shell local (via celulas Python com subprocess).

### Quando usar MCP no chat

Use prompts diretos no chat para o agente executar os passos:

1. Commit unico

```text
Fazer commit das alteracoes atuais com mensagem: feat: descricao objetiva
```

2. Commit por escopo

```text
Criar 2 commits separados: primeiro docs e depois codigo.
```

3. Abrir PR

```text
Abrir PR da branch minha-branch para main com titulo "feat: ..." e descricao resumindo contexto, alteracoes e validacao.
```

4. Sync da branch com main

```text
Sincronizar minha branch atual com main usando rebase e reportar conflitos.
```

5. Sync de labels (quando `.github/labels.yml` mudar)

```text
Executar o sync de labels do repositorio.
```

### Observacoes

- O auto-label de PR usa `/.github/labeler.yml` e workflow `/.github/workflows/pr-labeler.yml`.
- O sync de labels usa `/.github/workflows/sync-labels.yml`.
- Para PR por CLI, normalmente e necessario `gh auth login` previamente.


In [3]:
# O que faz: fornece funcoes utilitarias para inspecionar git status, criar commit, sincronizar branch e abrir PR via CLI.
# Onde executa: shell local do repositorio, a partir deste notebook.
from pathlib import Path
import subprocess


def run(command: str, cwd: Path | None = None, check: bool = True) -> subprocess.CompletedProcess:
    repo_root = cwd or Path(subprocess.check_output(["git", "rev-parse", "--show-toplevel"], text=True).strip())
    print(f"$ {command}")
    proc = subprocess.run(command, cwd=repo_root, shell=True, text=True, capture_output=True)
    if proc.stdout:
        print(proc.stdout.strip())
    if proc.stderr:
        print(proc.stderr.strip())
    if check and proc.returncode != 0:
        raise RuntimeError(f"Comando falhou com codigo {proc.returncode}: {command}")
    return proc


def git_status_short() -> None:
    run("git status --short")


def git_commit_all(message: str) -> None:
    run("git add -A")
    run(f"git commit -m \"{message}\"")
    run("git --no-pager log -1 --oneline")


def git_sync_with_main_rebase() -> None:
    run("git fetch origin")
    run("git rebase origin/main")


def gh_open_pr(base: str = "main", title: str = "", body: str = "") -> None:
    if not title:
        raise ValueError("title e obrigatorio")
    if not body:
        raise ValueError("body e obrigatorio")
    run(f"gh pr create --base {base} --title \"{title}\" --body \"{body}\"")


print("Helpers carregados. Exemplo rapido:")
print("- git_status_short()")
print("- git_commit_all('feat: exemplo')")
print("- git_sync_with_main_rebase()")
print("- gh_open_pr(title='feat: ...', body='contexto/alteracoes/validacao')")


Helpers carregados. Exemplo rapido:
- git_status_short()
- git_commit_all('feat: exemplo')
- git_sync_with_main_rebase()
- gh_open_pr(title='feat: ...', body='contexto/alteracoes/validacao')
